# Meijer Client - Shopping Cart Management

This notebook covers **working** shopping cart functionality:
- ✅ Viewing cart contents
- ✅ Adding items by UPC
- ✅ Updating quantities
- ✅ Removing items
- ✅ Setting store for cart operations
- ✅ Complete cart workflow demonstration

**Note**: This notebook demonstrates actual cart functionality based on real API endpoints found in mitmproxy logs.


## Import and Setup

In [ ]:
# Import required modules
from meijer.client import Meijer
import logging

# Configure logging to see what's happening
logging.basicConfig(level=logging.INFO)

# Initialize client
client = Meijer()
print("✅ Client initialized")

## Check Cart Availability

In [ ]:
# Check if cart functionality is available
if hasattr(client, "cart") and client.cart:
    print("✅ Cart functionality available")
    print(f"🏪 Current store: {client.cart.store_id}")
    print(
        f"🔄 Cart refresh method: {'Available' if hasattr(client.cart, 'refresh') else 'Not available'}"
    )
    print(
        f"➕ Add item method: {'Available' if hasattr(client.cart, 'add_item_by_upc') else 'Not available'}"
    )
    print(
        f"🔄 Update quantity method: {'Available' if hasattr(client.cart, 'update_item_quantity') else 'Not available'}"
    )
    print(
        f"🗑️ Remove item method: {'Available' if hasattr(client.cart, 'remove_item') else 'Not available'}"
    )
else:
    print("❌ Cart functionality not available")
    print("💡 This might be due to:")
    print("   - Client not properly initialized")
    print("   - Cart module not loaded")
    print("   - Authentication issues")

## View Current Cart

In [ ]:
# Get current cart contents
try:
    if hasattr(client, "cart") and client.cart:
        cart_items = client.cart.items

        if cart_items:
            print(f"🛒 Cart contains {len(cart_items)} items:")
            print()

            for i, item in enumerate(cart_items, 1):
                name = getattr(item, "name", f"Item {i}")
                qty = getattr(item, "current_quantity", getattr(item, "quantity", 1))
                price = getattr(item, "price", 0.0)
                entry_num = getattr(item, "entry_number", "N/A")

                print(f"  {i}. {name}")
                print(f"     Quantity: {qty}")
                print(f"     Price: ${price:.2f}")
                print(f"     Entry #: {entry_num}")
                print()
        else:
            print("🛒 Cart is empty")

        # Show cart totals
        if hasattr(client.cart, "total_quantity"):
            print(f"📊 Total quantity: {client.cart.total_quantity}")
        if hasattr(client.cart, "total_price"):
            print(f"💰 Total price: ${client.cart.total_price:.2f}")

    else:
        print("❌ Cart functionality not available")

except Exception as e:
    print(f"❌ Failed to get cart: {e}")
    print("💡 This might be due to:")
    print("   - Authentication issues")
    print("   - API endpoint not accessible")
    print("   - Network connectivity problems")

## Add Item to Cart

In [ ]:
# Add an item to cart by UPC
def add_item_to_cart(upc: str, quantity: int = 1):
    """Add an item to cart by UPC code."""
    try:
        if not hasattr(client, "cart") or not client.cart:
            print("❌ Cart functionality not available")
            return False

        print(f"🔄 Adding {quantity}x item with UPC {upc}...")
        success = client.cart.add_item_by_upc(upc, quantity)

        if success:
            print(f"✅ Successfully added {quantity}x item with UPC {upc}")

            # Refresh cart to show updated contents
            try:
                client.cart.refresh()
                print("🔄 Cart refreshed")
                return True
            except Exception as e:
                print(f"⚠️ Could not refresh cart: {e}")
                return True  # Item was added, just couldn't refresh
        else:
            print(f"❌ Failed to add item with UPC {upc}")
            print("💡 This might be due to:")
            print("   - Invalid UPC code")
            print("   - Item not available at current store")
            print("   - API endpoint not accessible")
            print("   - Authentication issues")
            return False

    except Exception as e:
        print(f"❌ Error adding item to cart: {e}")
        return False


# Example: Add a test item (replace with real UPC)
# add_item_to_cart("0000000000000", 2)

# Test with a real UPC (uncomment and modify as needed)
# add_item_to_cart("0000000000000", 1)

## Update Item Quantity

In [ ]:
# Update the quantity of an item in cart
def update_item_quantity(item_index: int, new_quantity: int):
    """Update the quantity of an item in cart."""
    try:
        if not hasattr(client, "cart") or not client.cart:
            print("❌ Cart functionality not available")
            return False

        cart_items = client.cart.items
        if not cart_items:
            print("❌ Cart is empty - no items to update")
            return False

        if item_index < 1 or item_index > len(cart_items):
            print(
                f"❌ Invalid item index {item_index}. Cart has {len(cart_items)} items."
            )
            return False

        item = cart_items[item_index - 1]
        entry_number = getattr(item, "entry_number", None)
        current_qty = getattr(item, "current_quantity", getattr(item, "quantity", 1))
        item_name = getattr(item, "name", f"Item {item_index}")

        if not entry_number:
            print(f"❌ Cannot update {item_name} - no entry number available")
            return False

        print(f"🔄 Updating {item_name} from quantity {current_qty} to {new_quantity}")

        success = client.cart.update_item_quantity(entry_number, new_quantity)

        if success:
            print(f"✅ Successfully updated {item_name} quantity to {new_quantity}")

            # Refresh cart to show updated contents
            try:
                client.cart.refresh()
                print("🔄 Cart refreshed")
                return True
            except Exception as e:
                print(f"⚠️ Could not refresh cart: {e}")
                return True  # Quantity was updated, just couldn't refresh
        else:
            print(f"❌ Failed to update {item_name} quantity")
            return False

    except Exception as e:
        print(f"❌ Error updating item quantity: {e}")
        return False


# Example: Update first item to quantity 5 (uncomment when cart has items)
# update_item_quantity(1, 5)

## Remove Item from Cart

In [ ]:
# Remove an item from cart
def remove_item_from_cart(item_index: int):
    """Remove an item from cart."""
    try:
        if not hasattr(client, "cart") or not client.cart:
            print("❌ Cart functionality not available")
            return False

        cart_items = client.cart.items
        if not cart_items:
            print("❌ Cart is empty - no items to remove")
            return False

        if item_index < 1 or item_index > len(cart_items):
            print(
                f"❌ Invalid item index {item_index}. Cart has {len(cart_items)} items."
            )
            return False

        item = cart_items[item_index - 1]
        entry_number = getattr(item, "entry_number", None)
        item_name = getattr(item, "name", f"Item {item_index}")

        if not entry_number:
            print(f"❌ Cannot remove {item_name} - no entry number available")
            return False

        print(f"🗑️ Removing {item_name} from cart")

        success = client.cart.remove_item(entry_number)

        if success:
            print(f"✅ Successfully removed {item_name} from cart")

            # Refresh cart to show updated contents
            try:
                client.cart.refresh()
                print("🔄 Cart refreshed")
                return True
            except Exception as e:
                print(f"⚠️ Could not refresh cart: {e}")
                return True  # Item was removed, just couldn't refresh
        else:
            print(f"❌ Failed to remove {item_name} from cart")
            return False

    except Exception as e:
        print(f"❌ Error removing item from cart: {e}")
        return False


# Example: Remove first item (uncomment when cart has items)
# remove_item_from_cart(1)

## Set Store for Cart Operations

In [ ]:
# Set the store for cart operations
def set_cart_store(store_id: str):
    """Set the store for cart operations."""
    try:
        if not hasattr(client, "cart") or not client.cart:
            print("❌ Cart functionality not available")
            return False

        old_store = client.cart.store_id
        print(f"🔄 Changing store from {old_store} to {store_id}")

        client.cart.store_id = store_id

        print(f"✅ Store changed to {client.cart.store_id}")
        print("💡 Cart data will be refreshed for the new store")

        # Clear cart cache for new store
        try:
            client.cart.refresh()
            print("🔄 Cart refreshed for new store")
            return True
        except Exception as e:
            print(f"⚠️ Could not refresh cart: {e}")
            return True  # Store was changed, just couldn't refresh

    except Exception as e:
        print(f"❌ Error setting store: {e}")
        return False


# Example: Set store to 217 (default)
# set_cart_store("217")

# Example: Set store to a different location
# set_cart_store("123")

## Complete Cart Workflow Example

In [ ]:
# Complete cart workflow demonstration
def demonstrate_cart_workflow():
    """Demonstrate a complete cart workflow."""
    print("🔄 === COMPLETE CART WORKFLOW DEMO ===")

    try:
        # 1. Set store
        print("1️⃣ Setting store to 217 (default)")
        set_cart_store("217")

        # 2. Show initial cart
        print("\n2️⃣ Showing initial cart contents")
        # View current cart (code from above)

        # 3. Add a test item (using a common UPC)
        test_upc = "0000000000000"  # Placeholder UPC
        print(f"\n3️⃣ Adding test item with UPC {test_upc}")
        add_item_to_cart(test_upc, 2)

        # 4. Show updated cart
        print("\n4️⃣ Showing updated cart contents")
        # View current cart (code from above)

        # 5. Update quantity
        print("\n5️⃣ Updating item quantity")
        update_item_quantity(1, 5)

        # 6. Show final cart
        print("\n6️⃣ Showing final cart contents")
        # View current cart (code from above)

        # 7. Remove item
        print("\n7️⃣ Removing item from cart")
        remove_item_from_cart(1)

        # 8. Show final cart
        print("\n8️⃣ Showing final cart contents")
        # View current cart (code from above)

        print("\n✅ Cart workflow demo completed!")

    except Exception as e:
        print(f"❌ Error in cart workflow demo: {e}")


# Uncomment to run the complete workflow demo
# demonstrate_cart_workflow()

## Cart Management Tips

In [ ]:
# Tips for successful cart management:
print("💡 Cart Management Tips:")
print("1. Always authenticate first with 'meijer login'")
print("2. Set the correct store ID for your location")
print("3. Use valid UPC codes for adding items")
print("4. Check item availability at your selected store")
print("5. Monitor API responses for error messages")
print("6. Use the refresh() method to get latest cart data")
print("7. Handle entry numbers carefully for updates/removals")
print("8. Test with small quantities first")
print("9. Keep track of entry numbers for cart operations")
print("10. Use the CLI commands for quick cart management")

## Troubleshooting Common Issues

In [ ]:
# Common cart issues and solutions
print("🔧 Troubleshooting Common Cart Issues:")
print()
print("❌ 'Cart functionality not available'")
print("   → Check if client.cart exists and is initialized")
print("   → Ensure proper authentication")
print("   → Verify cart module is loaded")
print()
print("❌ 'Failed to add item'")
print("   → Verify UPC code is valid")
print("   → Check if item is available at current store")
print("   → Ensure store ID is set correctly")
print("   → Check authentication status")
print()
print("❌ 'Failed to update quantity'")
print("   → Verify item exists in cart")
print("   → Check if entry_number is available")
print("   → Ensure quantity is >= 1")
print("   → Verify item is still available")
print()
print("❌ 'Failed to remove item'")
print("   → Verify item exists in cart")
print("   → Check if entry_number is available")
print("   → Ensure proper authentication")
print("   → Check if item was already removed")
print()
print("❌ 'API endpoint not accessible'")
print("   → Check network connectivity")
print("   → Verify API base URL is correct")
print("   → Check if endpoints have changed")
print("   → Ensure proper headers are sent")

## CLI Commands Reference

In [ ]:
# Available CLI commands for cart management
print("🖥️ Available CLI Commands:")
print()
print("📋 View cart contents:")
print("   meijer cart show")
print()
print("➕ Add item to cart:")
print("   meijer cart add <UPC> [--quantity <qty>] [--store <store_id>]")
print()
print("🔄 Update item quantity:")
print("   meijer cart update <index> <quantity>")
print()
print("🗑️ Remove item from cart:")
print("   meijer cart remove <index> [--quantity <qty>]")
print()
print("🏪 Set store for cart operations:")
print("   meijer cart set-store <store_id>")
print()
print("🧹 Clear entire cart:")
print("   meijer cart clear")
print()
print("ℹ️ Show cart information:")
print("   meijer cart info")
print()
print("📅 Show delivery/pickup slots:")
print("   meijer cart slots")
print()
print("💳 Proceed to checkout:")
print("   meijer cart checkout")

## Testing Cart Functionality

In [ ]:
# Test cart functionality step by step
print("🧪 Testing Cart Functionality Step by Step")
print("=" * 50)

# Step 1: Check cart availability
print("\n1️⃣ Testing cart availability...")
if hasattr(client, "cart") and client.cart:
    print("   ✅ Cart is available")
else:
    print("   ❌ Cart is not available")
    print("   💡 Check client initialization and authentication")

# Step 2: Check cart methods
if hasattr(client, "cart") and client.cart:
    print("\n2️⃣ Testing cart methods...")
    methods = ["add_item_by_upc", "remove_item", "update_item_quantity", "refresh"]
    for method in methods:
        if hasattr(client.cart, method):
            print(f"   ✅ {method}: Available")
        else:
            print(f"   ❌ {method}: Not available")

# Step 3: Check cart properties
if hasattr(client, "cart") and client.cart:
    print("\n3️⃣ Testing cart properties...")
    properties = ["items", "store_id", "total_quantity", "total_price"]
    for prop in properties:
        if hasattr(client.cart, prop):
            print(f"   ✅ {prop}: Available")
        else:
            print(f"   ❌ {prop}: Not available")

print("\n✅ Cart functionality testing completed!")